# EU CRR RAG â€” GPU Dev Environment

Run on a **T4 GPU** runtime (Runtime â†’ Change runtime type â†’ T4 GPU).

This notebook covers:
- **Cells 0â€“4**: Setup (clone, deps, secrets, HTML files)
- **Cells 5â€“8**: Ingestion (EN + IT, ~5 min on T4)
- **Cell 9**: Smoke-test query
- **Cells 10â€“12**: Integration tests (run per-file)
- **Cell 13**: Ad-hoc queries

> Unit tests run automatically on every push via GitHub Actions â€” no GPU needed for those.

## 0 â€” Verify GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    raise RuntimeError('No GPU detected. Go to Runtime â†’ Change runtime type â†’ T4 GPU, then reconnect.')

In [ ]:
# ── Experiment config ───────────────────────────────────────────────────────
# Switch EMBED_MODEL to run A/B experiments without touching code.
#   "bge-m3"           → BAAI/bge-m3 (default/production, 1024-dim dense+sparse hybrid)
#   "e5-large-instruct" → multilingual-e5-large-instruct (1024-dim dense only, A/B only)
# Set QDRANT_COLLECTION to a new name (e.g. eu_crr_e5) to ingest into a
# separate collection so the production index is never touched.
# ─────────────────────────────────────────────────────────────────────────────
import os
EMBED_MODEL       = "bge-m3"  # "bge-m3" (production) | "e5-large-instruct" (A/B experiment)
QDRANT_COLLECTION = "eu_crr"  # "eu_crr" (production) | "eu_crr_e5" (A/B experiment)

os.environ["EMBED_MODEL"]       = EMBED_MODEL
os.environ["QDRANT_COLLECTION"] = QDRANT_COLLECTION
print(f"Embed model : {EMBED_MODEL}")
print(f"Collection  : {QDRANT_COLLECTION}")

## 1 â€” Clone the repo

In [ ]:
import os

REPO_DIR = "/content/eu-crr-rag"
BRANCH = "main"

if not os.path.exists(REPO_DIR):
    !git clone --branch {BRANCH} https://github.com/apolitano20/EU-CRR-RAG.git {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull origin {BRANCH}

%cd {REPO_DIR}
print('Working directory:', os.getcwd())
print('Branch:', BRANCH)

## 2 â€” Install dependencies

Colab already ships with a CUDA-enabled PyTorch â€” do **not** reinstall it.
This cell only installs the project-specific packages (~2â€“5 min, BGE-M3 model is ~570 MB).

In [ ]:
# Install from requirements files (Colab's CUDA PyTorch is already correct — do NOT reinstall torch)
!pip install -q -r requirements.txt -r requirements-dev.txt
# LlamaIndex's Settings lazy resolver tries to import this as the default embed model
# even though we use BGE-M3. Without it the resolver raises ImportError at engine.load().
!pip install -q llama-index-embeddings-openai
print('Install complete.')

## 3 â€” API keys

Set secrets via the ðŸ”‘ key icon in the left sidebar (recommended), then run this cell.
Alternatively, paste values directly â€” but do **not** save the notebook with keys in it.

In [ ]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["QDRANT_URL"]     = userdata.get("QDRANT_URL")
os.environ["QDRANT_API_KEY"] = userdata.get("QDRANT_API_KEY")
os.environ["USE_RERANKER"]   = "true"  # T4 has enough VRAM for any ~560MB model

with open(".env", "w") as f:
    for k in ("OPENAI_API_KEY", "QDRANT_URL", "QDRANT_API_KEY"):
        f.write(f"{k}={os.environ[k]}
")
    f.write("USE_RERANKER=true
")
    f.write("USE_PARAGRAPH_CHUNKING=true
")
    f.write(f"EMBED_MODEL={EMBED_MODEL}
")
    f.write(f"QDRANT_COLLECTION={QDRANT_COLLECTION}
")

print(f".env written. EMBED_MODEL={EMBED_MODEL}, QDRANT_COLLECTION={QDRANT_COLLECTION}")

## 4 â€” Upload HTML files

Upload `crr_raw_en.html` and `crr_raw_ita.html` using the Colab file browser **or** run the cell below to upload interactively.

In [ ]:
from google.colab import files
import shutil, os

print("Select your HTML files (crr_raw_en.html and crr_raw_ita.html):")
uploaded = files.upload()

for fname in uploaded:
    dest = os.path.join(REPO_DIR, fname)
    shutil.move(fname, dest)
    print(f"Moved {fname} â†’ {dest}")

for expected in ["crr_raw_en.html", "crr_raw_ita.html"]:
    path = os.path.join(REPO_DIR, expected)
    if os.path.exists(path):
        print(f"  âœ“ {expected} ({os.path.getsize(path)/1e6:.1f} MB)")
    else:
        print(f"  âœ— {expected} NOT FOUND â€” upload it before continuing")

## 5 â€” Warm up BGE-M3

Downloads the model (~570 MB) on first run. Confirms it is running on the GPU.

In [ ]:
import os

if os.getenv("EMBED_MODEL", "bge-m3") == "e5-large-instruct":
    # E5-large-instruct: loaded via sentence-transformers on demand at ingest time.
    # Warm it up here to confirm GPU placement and download (~560 MB on first run).
    import torch
    from sentence_transformers import SentenceTransformer
    print("Loading multilingual-e5-large-instruct (~560 MB on first run)...")
    m = SentenceTransformer("intfloat/multilingual-e5-large-instruct")
    if torch.cuda.is_available():
        m = m.to("cuda")
    device = next(m.parameters()).device
    print(f"Model loaded on: {device}")
    if str(device) == "cpu":
        raise RuntimeError("Model is on CPU — check T4 GPU runtime is active.")
    test = m.encode(["passage: Article 92 own funds requirements"], normalize_embeddings=True)
    print(f"Dense embedding dim: {len(test[0])}")
    print("E5-large-instruct OK.")
else:
    # BGE-M3: default model, 570 MB.
    import torch
    from FlagEmbedding import BGEM3FlagModel
    print("Loading BGE-M3 (first run downloads ~570 MB)...")
    model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True)
    if torch.cuda.is_available():
        model.model = model.model.to("cuda")
    device = next(model.model.parameters()).device
    print(f"Model loaded on: {device}")
    if str(device) == "cpu":
        raise RuntimeError("Model is on CPU — check T4 GPU runtime is active.")
    test_out = model.encode(["Article 92 own funds requirements"], return_dense=True, return_sparse=True)
    print(f"Dense embedding dim: {len(test_out["dense_vecs"][0])}")
    print("BGE-M3 OK.")

## 6 — Ingest English

`--reset` wipes the entire Qdrant collection first.
With paragraph chunking, articles with ≥2 numbered paragraphs produce extra PARAGRAPH docs — expect **>745** items after this step.

In [ ]:
!python -m src.pipelines.ingest_pipeline \
    --reset \
    --language en \
    --file crr_raw_en.html

## 7 — Ingest Italian

No `--reset` — appends to the existing collection.
Expect total count **>1490** after this step (EN + IT ARTICLE + PARAGRAPH docs).

In [ ]:
!python -m src.pipelines.ingest_pipeline \
    --language it \
    --file crr_raw_ita.html

## 8 — Validate

Paragraph chunking produces ARTICLE + PARAGRAPH docs per article.
Expected item count: **>1490** (was 1490 with article-only index).

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from src.indexing.vector_store import VectorStore

vs = VectorStore()
vs.connect()
count = vs.item_count
print(f"Items in Qdrant collection: {count}")

# With paragraph chunking: ARTICLE docs + PARAGRAPH docs = >1490 total
if count > 1490:
    print("PASS -- paragraph-chunked index has more than article-only baseline (>1490)")
elif count >= 1480:
    print("PARTIAL -- count looks like article-only index; check USE_PARAGRAPH_CHUNKING=true was set")
elif count >= 740:
    print("PARTIAL -- only one language ingested; re-run Step 7 for Italian")
else:
    print(f"FAIL -- unexpectedly low count ({count}); check logs above")

# Spot-check: Article 92 EN should have both ARTICLE and PARAGRAPH docs
payloads = vs.scroll_payloads(language="en")
art92 = [p for p in payloads if p.get("article") == "92"]
art92_article = [p for p in art92 if p.get("chunk_type") == "ARTICLE"]
art92_para = [p for p in art92 if p.get("chunk_type") == "PARAGRAPH"]
print(f"\nArticle 92 EN: {len(art92_article)} ARTICLE doc(s), {len(art92_para)} PARAGRAPH doc(s)")
if art92_article and art92_para:
    print("PASS -- dual-document index confirmed for Article 92")
else:
    print("WARN -- unexpected structure; check ingestion logs")

## 8b â€” Detailed diagnosis

Breaks down items per language, checks for duplicate node_ids, and lists annex sub-items.
All counts should be clean after the deterministic `id_=node.node_id` fix (2026-03-17).

In [ ]:
!python scripts/diagnose_qdrant.py --no-parser

## 9 â€” Smoke-test a query

Verifies that the QueryEngine loads and returns a substantive answer citing article numbers.

In [ ]:
import os
from dotenv import load_dotenv; load_dotenv()

from src.indexing.vector_store import VectorStore
from src.indexing.index_builder import HierarchicalIndexer
from src.query.query_engine import QueryEngine

vector_store = VectorStore()
indexer = HierarchicalIndexer(vector_store=vector_store)
engine = QueryEngine(vector_store=vector_store, indexer=indexer, openai_api_key=os.environ["OPENAI_API_KEY"])
engine.load()
print(f"Engine ready. use_reranker={engine.use_reranker}")

result = engine.query("What are the own funds requirements under Article 92?", language="en")

print("=" * 60)
print(result.answer)
print("=" * 60)
print(f"\nSources: {len(result.sources)}")
for s in result.sources:
    m = s.get('metadata', {})
    print(f"  Article {m.get('article','?')} [{m.get('language','?')}]  score={s.get('score',0):.3f}  expanded={s.get('expanded')}")

## Done

The Qdrant collection is now populated with paragraph-chunked docs (EN + IT).
Each multi-paragraph article has one ARTICLE doc (synthesis context) + N PARAGRAPH docs (retrieval).
Your local API server will use this index on next start — set `USE_PARAGRAPH_CHUNKING=true` in `.env`.

In [ ]:
QUERY    = "What is the leverage ratio requirement under Article 429?"
LANGUAGE = "en"   # "en", "it", or None for cross-language

result = engine.query(QUERY, language=LANGUAGE)
print("=" * 60)
print(result.answer)
print("=" * 60)
for s in result.sources:
    m = s.get('metadata', {})
    print(f"  Article {m.get('article','?')} [{m.get('language','?')}]  score={s.get('score',0):.3f}")

---
## Ad-hoc Queries

Edit `QUERY` and `LANGUAGE` then run. Engine must be loaded from the smoke-test cell above.

In [ ]:
!pytest tests/integration/test_query_engine.py -v --tb=short

In [ ]:
!pytest tests/integration/test_ingest_pipeline.py -v --tb=short

In [ ]:
!pytest tests/integration/test_vector_store.py -v --tb=short

---
## Integration Tests

Run per-file â€” BGE-M3 segfaults if all three run in one pytest invocation.